# Squad Optimizer — ILP under Fantasy Constraints

Picks the 15-player squad that **maximizes total predicted UCL fantasy points**, subject to the real constraints of the game:

- €100m budget
- Exactly 2 GK, 5 DEF, 5 MID, 3 FWD
- Max 3 players from any one club

## Hybrid model: each position uses its own best-fitting window

`ols_fantasy_model.ipynb` Section 5 found that "more history helps" is **not** a universal rule — it depends on the position:

| Position | Best k | Adj. R² at that k | Why |
|---|---|---|---|
| GK | k=1 | 0.157 | Adj. R² gets *worse* every time more history is added (0.157→0.097→0.058→0.053) |
| DEF | k=4 | 0.140 | Adj. R² steadily *improves* with more history (0.062→0.128→0.137→0.140) — single-season defensive stats are noisy, multi-season averaging smooths that out |
| MID | k=1 | 0.189 | Same pattern as GK — more history steadily hurts (0.189→0.155→0.152→0.141) |
| FWD | k=4 | 0.336 | Improves like DEF (0.310→0.314→0.326→0.336), though more mildly |

So this notebook uses the **k=1 model for GK/MID** (their single most recent domestic season) and the **k=4 model for DEF/FWD** (their trailing 4-season domestic average) — each position genuinely scored by whichever version of the model actually fits it best, rather than applying one window uniformly.

## What is an ILP, and how does it decide?

**ILP = Integer Linear Programming.** One binary decision variable per player (`x_i ∈ {0,1}`: in the squad or not), an objective to maximize (`sum(x_i × predicted_points_i)`), and a set of linear constraints the solution must satisfy exactly (budget, position counts, club limit).

This is **not** a greedy heuristic ("grab the best player, then the next-best that still fits") — greedy approaches can get stuck in a bad combination, e.g. blowing the budget on one star early and leaving nothing for a strong supporting cast. A real ILP solver (here: CBC, via the `PuLP` library) explores the combination space systematically using branch-and-bound, and **proves** it found the mathematically best possible total given the objective and constraints — not just a good guess. That's what `Status: Optimal` means below.

The one thing it can't do: know if the underlying model is wrong about a player. It will ruthlessly and correctly maximize *predicted* points — garbage in, garbage out still applies. All the model-correctness work (fixing the lag, checking coefficients, the k-window analysis, and now this hybrid best-k-per-position choice) is what makes the objective worth maximizing in the first place; this notebook is just the mechanical last step.

## Repo layout
- `data/` — inputs this notebook reads
- Run this notebook from the project root (`fantasy ucl/`) so the relative `data/` path resolves.

## Data used
`data/squad_optimizer_input_hybrid.csv` — every current UEFA Fantasy player with their price, position, club, and each player's predicted 2026/27 UCL fantasy points using their position's best-fitting k (already computed and joined here, so this notebook doesn't need to redo the name-matching or multi-season averaging).

In [1]:
import pandas as pd
import pulp

players = pd.read_csv('data/squad_optimizer_input_hybrid.csv')
print(f"Total players: {len(players)}  |  Matched to a prediction: {players['Matched'].sum()}")
print(players.groupby('Position')['KUsed'].first())
players.head()

Total players: 1163  |  Matched to a prediction: 893
Position
DEF    4
FWD    4
GK     1
MID    1
Name: KUsed, dtype: int64


,Id,Name,Position,Price,ClubId,PredictedPts_BestK,KUsed,Matched
0,250076574,K. Mbappé,FWD,11.0,50051,38.35,4,True
1,250103758,E. Haaland,FWD,11.0,52919,35.64,4,True
2,250016833,H. Kane,FWD,11.0,50037,40.70,4,True
3,250176450,L. Yamal,MID,10.0,50080,28.50,1,True
4,250101808,K. Kvaratskhelia,MID,10.0,52747,15.28,1,True


## 1. Set up the ILP

In [2]:
BUDGET = 100.0
POSITION_QUOTAS = {'GK': 2, 'DEF': 5, 'MID': 5, 'FWD': 3}
MAX_PER_CLUB = 3

prob = pulp.LpProblem('squad_optimizer', pulp.LpMaximize)

# one binary variable per player: 1 = selected, 0 = not
x = {row.Id: pulp.LpVariable(f'x_{row.Id}', cat='Binary') for row in players.itertuples()}

# objective: maximize total predicted points (each position's own best-fitting k)
prob += pulp.lpSum(x[row.Id] * row.PredictedPts_BestK for row in players.itertuples())

# constraint: budget
prob += pulp.lpSum(x[row.Id] * row.Price for row in players.itertuples()) <= BUDGET

# constraint: exact position quotas
for pos, quota in POSITION_QUOTAS.items():
    prob += pulp.lpSum(x[row.Id] for row in players.itertuples() if row.Position == pos) == quota

# constraint: max players per club
for club_id in players['ClubId'].unique():
    prob += pulp.lpSum(x[row.Id] for row in players.itertuples() if row.ClubId == club_id) <= MAX_PER_CLUB

status = prob.solve(pulp.PULP_CBC_CMD(msg=0))
print('Status:', pulp.LpStatus[prob.status])

Status: Optimal


## 2. The optimal squad

In [3]:
chosen_ids = [pid for pid, var in x.items() if var.value() == 1]
squad = players[players['Id'].isin(chosen_ids)].copy()

pos_order = pd.CategoricalDtype(['GK', 'DEF', 'MID', 'FWD'], ordered=True)
squad['Position'] = squad['Position'].astype(pos_order)
squad = squad.sort_values(['Position', 'Price'], ascending=[True, False])

print(f"Total spent: €{squad['Price'].sum():.1f}m / €{BUDGET}m")
print(f"Total predicted points: {squad['PredictedPts_BestK'].sum():.2f}")
squad[['Position', 'Name', 'Price', 'PredictedPts_BestK', 'KUsed']].reset_index(drop=True)

Total spent: €99.5m / €100.0m
Total predicted points: 444.24


,Position,Name,Price,PredictedPts_BestK,KUsed
0,GK,Diogo Costa,5.0,24.85,1
1,GK,J. Butez,4.5,24.65,1
2,DEF,A. Grimaldo,5.5,31.85,4
3,DEF,F. Dimarco,5.0,24.35,4
4,DEF,B. Mechele,4.5,24.04,4
5,DEF,K. Jørgensen,4.5,24.61,4
6,DEF,N. Brown,4.0,23.26,4
7,MID,M. Olise,9.0,32.05,1
8,MID,Bruno Fernandes,9.0,30.59,1
9,MID,C. Tzolis,7.5,33.99,1


## 3. Sanity checks — confirm every constraint actually held

In [4]:
print('Position counts:')
print(squad['Position'].value_counts())
print()
print('Players per club (max allowed: 3):')
club_counts = squad['ClubId'].value_counts()
print(club_counts[club_counts > 1] if (club_counts > 1).any() else 'No club has more than 1 player selected')
assert squad['Price'].sum() <= BUDGET, 'Budget constraint violated!'
for pos, quota in POSITION_QUOTAS.items():
    actual = (squad['Position'] == pos).sum()
    assert actual == quota, f'{pos}: expected {quota}, got {actual}'
assert (club_counts <= MAX_PER_CLUB).all(), 'Club limit violated!'
print('\nAll constraints satisfied.')

Position counts:
Position
DEF    5
MID    5
FWD    3
GK     2
Name: count, dtype: int64

Players per club (max allowed: 3):
ClubId
50037    3
52280    2
Name: count, dtype: int64

All constraints satisfied.


## 4. Which of your current squad survive?

Update `MY_CURRENT_SQUAD_IDS` below with your actual UEFA Fantasy player IDs (from the `Id` column) to see which of your existing picks the optimizer agrees with.

In [5]:
MY_CURRENT_SQUAD_IDS = {
    250131901, 250042422, 250112880, 250016833, 97923, 250113350, 250134856,
    250078886, 250075007, 250137260, 250082664, 250135068, 250136465, 250101444, 250112217,
}

overlap = squad[squad['Id'].isin(MY_CURRENT_SQUAD_IDS)]
print(f'{len(overlap)} of your current 15 players are in the optimal squad:')
overlap[['Position', 'Name', 'Price', 'PredictedPts_BestK']]

3 of your current 15 players are in the optimal squad:


,Position,Name,Price,PredictedPts_BestK
265,DEF,A. Grimaldo,5.5,31.85
11,MID,Bruno Fernandes,9.0,30.59
2,FWD,H. Kane,11.0,40.70


## 5. Caveats carried over from the model itself

- **270 players (1,163 − 893 matched)** have no prediction — either no clean name-match to Sofascore, or missing domestic data in the relevant window — and are excluded from consideration entirely.
- DEF/FWD predictions use a **trailing 4-season average**, so a player who only recently arrived in a top-flight league (fewer than 4 seasons of data available) may be scored on fewer seasons than intended, or excluded if none are available.
- Still no age, domestic cups, current-season data, fixture difficulty, or transfer/new-club adjustment — see `ols_fantasy_model.ipynb` Section 7 for the full list.